In [ ]:
# ── 1. IMPORTS & SETUP ────────────────────────────────────────
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv(override=True)
client = OpenAI()
MODEL  = "gpt-4o-mini"

In [ ]:
# ── 2. REAL-WORLD TOOL FUNCTIONS ──────────────────────────────
# These do actual work — the LLM decides WHEN to call them

def get_weather(city: str) -> dict:
    # In production: call a real weather API
    weather_data = {
        "Lagos":  {"temp": "32°C", "condition": "Sunny"},
        "London": {"temp": "14°C", "condition": "Cloudy"},
    }
    return weather_data.get(city, {"temp": "unknown", "condition": "unknown"})

def add_numbers(a: float, b: float) -> dict:
    return {"result": a + b}

In [ ]:
# ── 3. TOOL SCHEMAS ───────────────────────────────────────────
# JSON blueprints the LLM reads to understand WHAT tools exist
# description = WHEN to call it
# parameters  = WHAT to pass

get_weather_json = {
    "name": "get_weather",
    "description": "Get current weather for a city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city name e.g. Lagos, London"
            }
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

add_numbers_json = {
    "name": "add_numbers",
    "description": "Add two numbers together and return the result",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"}
        },
        "required": ["a", "b"],
        "additionalProperties": False
    }
}

# Register tools — passed to every LLM call
tools = [
    {"type": "function", "function": get_weather_json},
    {"type": "function", "function": add_numbers_json},
]

In [ ]:
# ── 4. TOOL HANDLER ───────────────────────────────────────────
# Executes whatever tools the LLM requests
# globals() maps "get_weather" string → actual get_weather function

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        name      = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        print(f"→ Calling tool: {name}({arguments})")

        fn     = globals().get(name)           # look up function by name
        result = fn(**arguments) if fn else {"error": "tool not found"}

        results.append({
            "role":         "tool",
            "content":      json.dumps(result),
            "tool_call_id": tool_call.id       # MUST match the request id
        })
    return results

In [ ]:
# ── 5. THE AGENT LOOP ─────────────────────────────────────────
# The core pattern — loops until LLM stops calling tools

def run_agent(user_message: str):
    print(f"\nUser: {user_message}\n")

    messages = [
        {"role": "system",  "content": "You are a helpful assistant with tools. Use them when needed."},
        {"role": "user",    "content": user_message}
    ]

    while True:
        response      = client.chat.completions.create(
            model    = MODEL,
            messages = messages,
            tools    = tools
        )
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            # LLM wants to use a tool
            assistant_msg = response.choices[0].message
            tool_calls    = assistant_msg.tool_calls

            # Execute the tools
            tool_results  = handle_tool_calls(tool_calls)

            # CRITICAL ORDER: assistant request THEN tool results
            messages.append(assistant_msg)
            messages.extend(tool_results)
            # Loop continues → LLM processes results

        else:
            # finish_reason == "stop" → LLM has final answer
            final_reply = response.choices[0].message.content
            print(f"\nAgent: {final_reply}")
            return final_reply

In [ ]:
run_agent("What's the weather in Lagos, and what is 847 + 253?")